In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

### Load and Process Data

In [ ]:
# 1. Load the data (pandas automatically uses the first row as the header)
file_path = '../results/my_test_results.csv'
df = pd.read_csv(file_path)

# 2. Calculate the metrics using your new column names
# Divide total time by the number of runs to get time per run
df['time_per_run'] = df['time_sec'] / df['runs']

# Calculate dif% from your 'avg_ratio' column
df['dif%'] = (df['avg_ratio'] - 1) * 100

# Extract the integer n from the dataset name (e.g., 'our-u-100' -> 100)
df['n'] = df['dataset'].str.extract(r'(\d+)').astype(int)

# 3. Create pivot tables for easy plotting
pivot_time = df.pivot(index='n', columns='algo', values='time_per_run')
pivot_dif = df.pivot(index='n', columns='algo', values='dif%')

# 4. Determine overall order from best to worst dif% to keep legends consistent
mean_dif = pivot_dif.mean().sort_values(ascending=True)
algos_ordered = list(mean_dif.index)

### Algorithm Runtimes

In [ ]:
# Define standard markers to easily distinguish the lines
markers = ['o', 's', '^', 'D', 'v', '<', '>', 'p']

# --- Graph 1: Percentage Difference (dif%) ---
plt.figure(figsize=(10, 6))
for i, algo in enumerate(algos_ordered):
    plt.plot(pivot_dif.index, pivot_dif[algo], marker=markers[i], linewidth=2, label=algo.upper())

plt.xlabel('Instance Size (n)', fontsize=12)
plt.ylabel('Percentage Difference (dif%)', fontsize=12)
plt.title('Solution Quality across Instance Sizes', fontsize=14)
# Place legend outside the plot so it does not overlap data
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

plt.savefig('solution_quality_plot.pdf', format='pdf', bbox_inches='tight')

plt.show()

In [ ]:
# --- Graph 2: Execution Time ---
plt.figure(figsize=(10, 6))
for i, algo in enumerate(algos_ordered):
    plt.plot(pivot_time.index, pivot_time[algo], marker=markers[i], linewidth=2, label=algo.upper())

plt.xlabel('Instance Size (n)', fontsize=12)
plt.ylabel('Execution Time (seconds)', fontsize=12)
plt.title('Execution Time across Instance Sizes', fontsize=14)
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

plt.savefig('execution_time_plot.pdf', format='pdf', bbox_inches='tight')

plt.show()

## Bar Charts

In [ ]:
def plot_bar_for_dataset(n_choice):
    # Check if the requested size exists in the data
    if n_choice not in pivot_dif.index:
        print(f"Dataset size {n_choice} not found. Available sizes: {list(pivot_dif.index)}")
        return
    
    # Extract and sort data for this specific dataset
    data_dif = pivot_dif.loc[n_choice].sort_values(ascending=True)
    data_time = pivot_time.loc[n_choice].sort_values(ascending=True)
    
    # Create side-by-side subplots
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
    
    # dif% Bar Chart
    ax1.bar(data_dif.index.str.upper(), data_dif.values, color='skyblue', edgecolor='black')
    ax1.set_title(f'Percentage Difference (dif%) for n={n_choice}', fontsize=14)
    ax1.set_ylabel('dif%', fontsize=12)
    ax1.set_xlabel('Algorithm', fontsize=12)
    ax1.grid(axis='y', linestyle='--', alpha=0.6)
    
    # Time Bar Chart
    ax2.bar(data_time.index.str.upper(), data_time.values, color='lightgreen', edgecolor='black')
    ax2.set_title(f'Execution Time (s) for n={n_choice}', fontsize=14)
    ax2.set_ylabel('Time (seconds)', fontsize=12)
    ax2.set_xlabel('Algorithm', fontsize=12)
    ax2.grid(axis='y', linestyle='--', alpha=0.6)
    
    # Rotate x-axis labels to prevent overlapping
    plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45, ha='right')
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()

# Generate charts for n=400
plot_bar_for_dataset(400)